# Variability Analysis for Nygaard19 (AN19) t-SNE Dataset (FT Model)

This notebook computes variability measures for the Nygaard (AN19) dataset, following the same logic as `xie_variability_tsne.ipynb` but adapted for the word-level structure of the AN19 stimuli.

## Key Adaptations from Xie (X21)
- **No sentence structure**: AN19 stimuli are isolated words, not sentences. Variability is computed at the word level.
- **Training talkers**: Each subject's training talkers are derived from their Learning phase filenames.
- **GLMM formula**: Uses the AN19 formula: `~ variability_scaled + (1 + variability_scaled | SubjectID)`.
- **Variability measures**:
  - `WithinWord`: Frame-level variance within each word (temporal dynamics)
  - `BetweenWord`: Variance of word-level means across the training lexicon
  - `Word_Order`: Consecutive frame difference variance (temporal smoothness)


In [ ]:
import pandas as pd
import numpy as np
import os
import h5py
import warnings
import traceback
from scipy.optimize import minimize_scalar
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold, KFold

os.environ['R_HOME'] = r'C:\Program Files\R\R-4.4.1'
os.environ["PATH"] += os.pathsep + r"C:\Program Files\R\R-4.4.1\bin\x64"
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
import rpy2.robjects as ro
pandas2ri.activate()
warnings.filterwarnings('ignore')

from project_utils import compute_jaeger_ceiling_nygaard


## 1. Data Loading and Fold Creation

Load the AN19 behavioral data and create 3-fold stratified splits. Build a subject→training-talkers map from the Learning phase.


In [ ]:
def create_nygaard_dataset(df_full):
    """
    Creates the stratified 3-fold split by grouping subjects with identical training/test files.
    """
    df_test = df_full[df_full["Phase"]=="Test"].copy().reset_index(drop=True)
    if 'TrainingAccent' not in df_test.columns and 'TrainingAccent' in df_full.columns:
        df_test['TrainingAccent'] = df_full.groupby('Subject')['TrainingAccent'].transform('first')
    elif 'TrainingAccent' not in df_full.columns:
        df_test['TrainingAccent'] = "English"
    df_test['TrainingAccent'] = df_test['TrainingAccent'].fillna("English")
    
    df_test_TrainingFile = []
    df_test_TestFile = []
    for index, row in df_test.iterrows():
        current_participant = row["Subject"]
        df_test_TrainingFile.append("_".join(sorted(list(df_full[(df_full["Subject"]==current_participant) & (df_full["Phase"]=="Learning")]["FileName"].dropna().astype(str)))))
        df_test_TestFile.append("_".join(sorted(list(df_full[(df_full["Subject"]==current_participant) & (df_full["Phase"]=="Test")]["FileName"].dropna().astype(str)))))
        
    df_test["TrainingFile"] = df_test_TrainingFile
    df_test["TestFile"] = df_test_TestFile
    df_test["TrainingFile"] = df_test["TrainingFile"].apply(lambda x: "English" if x=="" else x)
    df_test["TrainingTestFile"] = df_test["TrainingFile"] + df_test["TestFile"]
    
    participants_df = df_test[['Subject', 'TrainingTestFile']].drop_duplicates().reset_index(drop=True)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    participants_df['fold'] = -1
    try:
        split_generator = skf.split(participants_df, participants_df['TrainingTestFile'])
        for fold_idx, (train_index, test_index) in enumerate(split_generator):
            participants_df.loc[test_index, 'fold'] = fold_idx + 1
    except ValueError:
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        for fold_idx, (train_index, test_index) in enumerate(kf.split(participants_df)):
            participants_df.loc[test_index, 'fold'] = fold_idx + 1
            
    df_test = df_test.merge(participants_df[['Subject', 'fold']], on='Subject', how='left')
    if 'Speaker_full' not in df_test.columns:
        if 'Accent' in df_test.columns and 'Speaker' in df_test.columns:
            df_test["Speaker_full"] = df_test["Accent"].astype(str) + df_test["Speaker"].astype(str)
        else:
            df_test["Speaker_full"] = df_test["FileName"].apply(lambda x: str(x)[:3].lower() if pd.notna(x) else "unknown")
    return df_test


def get_training_talkers_map(df_full):
    """
    Build Subject -> [Training Speakers] map from the Learning phase.
    Each subject's training talkers are the unique speaker IDs (first 3 chars of FileName)
    from their Learning phase trials.
    """
    subject_map = {}
    default_control_group = ['ef1','ef2','ef3','em1','em2','em3']
    
    for sub in df_full['Subject'].unique():
        learning_phase = df_full[(df_full['Subject'] == sub) & (df_full['Phase'] == 'Learning')]
        if not learning_phase.empty:
            spks = learning_phase['FileName'].dropna().apply(lambda x: str(x)[:3].lower()).unique()
            subject_map[sub] = list(spks)
        else:
            subject_map[sub] = default_control_group
    return subject_map


def get_training_words_map(df_full):
    """
    Build Subject -> [Training Words] map from the Learning phase.
    Returns the word names each subject was exposed to during training.
    """
    subject_words = {}
    for sub in df_full['Subject'].unique():
        learning_phase = df_full[(df_full['Subject'] == sub) & (df_full['Phase'] == 'Learning')]
        if not learning_phase.empty:
            words = learning_phase['correct'].dropna().str.lower().unique()
            subject_words[sub] = list(words)
        else:
            subject_words[sub] = []
    return subject_words


# ── Load behavioral data ──
EXCEL_PATH = r"../data/raw_data/alexander_nygaard19/AN19-exposure-test-behavioral-data.xlsx"
print("Loading behavioral data...")
df_raw = pd.read_excel(EXCEL_PATH)
df_test = create_nygaard_dataset(df_raw)
subject_map = get_training_talkers_map(df_raw)
subject_words_map = get_training_words_map(df_raw)

print(f"  Test trials: {len(df_test)}")
print(f"  Subjects: {df_test['Subject'].nunique()}")
print(f"  Folds: {sorted(df_test['fold'].unique())}")
print(f"  Unique test speakers: {df_test['Speaker_full'].nunique()}")
print(f"  Sample subject_map: Subject 1 -> {subject_map.get(1, 'N/A')}")
print(f"  Sample subject_words: Subject 1 -> {subject_words_map.get(1, 'N/A')[:5]}... ({len(subject_words_map.get(1, []))} words)")


## 2. Feature Loading and Standardization

The AN19 HDF5 file is organized as `layer / speaker / word → (T, 3)`.
We load a single layer at a time and apply global z-score standardization.


In [ ]:
def load_nygaard_layer(h5_path, layer_key):
    """
    Load a single layer from the Nygaard HDF5 file.
    
    Returns
    -------
    layer_data : dict
        {speaker_id: {word_name: np.ndarray of shape (T, D)}}
    speakers : list
        List of all speaker IDs in the layer.
    """
    layer_data = {}
    with h5py.File(h5_path, 'r') as f:
        for spk in f[layer_key].keys():
            layer_data[spk] = {}
            for word in f[layer_key][spk].keys():
                layer_data[spk][word] = f[layer_key][spk][word][:]
    speakers = sorted(layer_data.keys())
    return layer_data, speakers


def standardize_nygaard_layer(layer_data):
    """
    Global z-score standardization across ALL frames in the layer.
    Modifies layer_data in-place and returns it.
    """
    # Collect all frames
    all_frames = []
    for spk in layer_data:
        for word in layer_data[spk]:
            all_frames.append(layer_data[spk][word])
    all_frames = np.vstack(all_frames)
    
    mu = all_frames.mean(axis=0, keepdims=True)
    sd = all_frames.std(axis=0, keepdims=True) + 1e-8
    
    # Apply standardization
    std_data = {}
    for spk in layer_data:
        std_data[spk] = {}
        for word in layer_data[spk]:
            std_data[spk][word] = (layer_data[spk][word] - mu) / sd
    return std_data


# ── Detect available layers ──
H5_PATH = r"../data/features/nygaard19_tsne_3d_ft.h5"

with h5py.File(H5_PATH, 'r') as f:
    layers_list = sorted(list(f.keys()))

print(f"Found {len(layers_list)} layers: {layers_list}")


## 3. Variability Computation

Three variability measures adapted for the word-level Nygaard structure:

1. **WithinWord**: Average generalized variance of frames within each word relative to the word mean.
   $$V_{\text{within}} = \frac{1}{|\text{words}|} \sum_{w} \frac{1}{T_w} \sum_{t=1}^{T_w} \sum_{d} |x_{t,d} - \mu_{w,d}|^{\tau}$$

2. **BetweenWord**: Variance of word-level mean representations across words.
   $$V_{\text{between}} = \frac{1}{|\text{words}|} \sum_{w} \sum_{d} |\mu_{w,d} - \mu_{\text{global},d}|^{\tau}$$

3. **Word_Order**: Average generalized variance of consecutive frame differences within words.
   $$V_{\text{order}} = \frac{1}{|\text{words}|} \sum_{w} \frac{1}{T_w-1} \sum_{t=2}^{T_w} \sum_{d} |x_{t,d} - x_{t-1,d}|^{\tau}$$


In [ ]:
def _generalized_variance(diff, tau):
    """Compute generalized variance: sum(|diff|^tau) per row."""
    if diff.ndim == 1:
        return np.sum(np.abs(diff) ** tau)
    return np.sum(np.abs(diff) ** tau, axis=1)


def compute_variability_for_condition_nygaard(training_talkers, word_list, std_data, tau, method):
    """
    Compute variability for one experimental condition (one subject's training set).
    
    Parameters
    ----------
    training_talkers : list of str
        Speaker IDs of the training talkers (e.g., ['kf1', 'kf3', 'km2']).
    word_list : list of str
        Words the subject was exposed to during training.
    std_data : dict
        {speaker: {word: ndarray(T, D)}} - standardized features.
    tau : float
        Exponent for generalized variance.
    method : str
        'WithinWord', 'BetweenWord', or 'Word_Order'.
    
    Returns
    -------
    float or np.nan
    """
    if method == 'WithinWord':
        # Average frame-level variance within each word
        token_variances = []
        for spk in training_talkers:
            if spk not in std_data:
                continue
            for word in word_list:
                if word not in std_data[spk]:
                    continue
                frames = std_data[spk][word]
                if frames.shape[0] == 0:
                    continue
                word_mean = np.mean(frames, axis=0)
                frame_vars = _generalized_variance(frames - word_mean, tau)
                token_variances.append(np.mean(frame_vars))
        return np.mean(token_variances) if token_variances else np.nan

    elif method == 'BetweenWord':
        # Variance of word-level means across the training lexicon
        word_means = []
        for spk in training_talkers:
            if spk not in std_data:
                continue
            for word in word_list:
                if word not in std_data[spk]:
                    continue
                frames = std_data[spk][word]
                if frames.shape[0] > 0:
                    word_means.append(np.mean(frames, axis=0))
        if not word_means:
            return np.nan
        global_mean = np.mean(word_means, axis=0)
        type_vars = [_generalized_variance(wm - global_mean, tau) for wm in word_means]
        return np.mean(type_vars)

    elif method == 'Word_Order':
        # Average consecutive frame difference variance
        token_variances = []
        for spk in training_talkers:
            if spk not in std_data:
                continue
            for word in word_list:
                if word not in std_data[spk]:
                    continue
                frames = std_data[spk][word]
                if frames.shape[0] < 2:
                    continue
                diffs = frames[1:] - frames[:-1]
                frame_pair_vars = _generalized_variance(diffs, tau)
                token_variances.append(np.mean(frame_pair_vars))
        return np.mean(token_variances) if token_variances else np.nan

    return np.nan


def precompute_layer_variability_nygaard(df, std_data, tau, method, subject_map, subject_words_map):
    """
    For each unique subject condition, compute variability and assign to every trial.
    
    Returns a DataFrame with all original columns + 'variability'.
    """
    # Cache variability per subject (same training talkers & words → same variability)
    var_cache = {}
    
    rows = []
    for row in df.itertuples(index=False):
        sub = row.Subject
        
        if sub not in var_cache:
            training_talkers = subject_map.get(sub, [])
            word_list = subject_words_map.get(sub, [])
            var_val = compute_variability_for_condition_nygaard(
                training_talkers, word_list, std_data, tau, method
            )
            var_cache[sub] = var_val
        
        rows.append({
            'correct': getattr(row, 'correct'),
            'Speaker_full': getattr(row, 'Speaker_full'),
            'Subject': sub,
            'accuracy': getattr(row, 'accuracy'),
            'variability': var_cache[sub],
            'fold': getattr(row, 'fold'),
        })
    
    return pd.DataFrame(rows)


## 4. GLMM Pipeline

The Nygaard variability GLMM follows the same random-effects structure as the Nygaard similarity GLMM:

```r
glmer(cbind(numCorrect, numIncorrect) ~ variability_scaled + (1 + variability_scaled | SubjectID),
      data = data, family = binomial(link = "logit"))
```

Pipeline per layer:
1. For each fold, optimize τ (tau) on the training set using `scipy.optimize.minimize_scalar`
2. Re-evaluate all folds with the mean optimal τ (corrected evaluation)
3. Collect z-test statistics across layers


In [ ]:
def run_glmm_variability_logic_nygaard(tau, train_df, test_df, purpose='optimize'):
    """
    Fit the Nygaard variability GLMM.
    
    Parameters
    ----------
    tau : float
        Not directly used in GLMM (already baked into variability).
    train_df : DataFrame
        Must contain 'correct', 'Speaker_full', 'Subject', 'accuracy', 'variability', 'fold'.
    test_df : DataFrame or None
        If purpose='evaluate', the test split.
    purpose : str
        'optimize' → return negative z_train for minimization.
        'evaluate' → return dict with z_train, z_test, log-likelihoods.
    """
    try:
        # Aggregate by (correct, Speaker_full, Subject) for GLMM
        train_agg = train_df.groupby(
            ['correct', 'Speaker_full', 'Subject'], as_index=False
        ).agg(
            variability=('variability', 'mean'),
            numCorrect=('accuracy', 'sum'),
            numWord=('accuracy', 'count')
        )
        train_agg['numIncorrect'] = (train_agg['numWord'] - train_agg['numCorrect']).clip(lower=0)
        train_agg = train_agg.dropna(subset=['variability'])
        
        # Gelman (2008) scaling: (x - mean) / (2 * sd)
        train_sd = train_agg['variability'].std()
        train_mean = train_agg['variability'].mean()
        if train_sd == 0 or pd.isna(train_sd):
            return 999.0 if purpose == 'optimize' else None
        
        train_agg['variability_scaled'] = (train_agg['variability'] - train_mean) / (2 * train_sd)
        
        # Rename for R
        train_agg.rename(columns={
            'correct': 'Keyword',
            'Speaker_full': 'TestTalker',
            'Subject': 'SubjectID'
        }, inplace=True)
        
        ro.globalenv['r_train'] = pandas2ri.py2rpy(train_agg)
        ro.r("""
            library(lme4)
            r_train$Keyword   <- factor(r_train$Keyword)
            r_train$TestTalker <- factor(r_train$TestTalker)
            r_train$SubjectID <- factor(r_train$SubjectID)
            model_train <- glmer(
                cbind(numCorrect, numIncorrect) ~ variability_scaled + (1 + variability_scaled | SubjectID),
                data=r_train, family=binomial(link="logit"),
                control=glmerControl(optimizer="bobyqa", optCtrl=list(maxfun=1e5)))
            z_train  <- summary(model_train)$coefficients[2, 3]
            ll_train <- as.numeric(logLik(model_train))
        """)
        z_train = ro.globalenv['z_train'][0]
        
        if purpose == 'optimize':
            return -z_train
        
        if purpose == 'evaluate' and test_df is not None:
            test_agg = test_df.groupby(
                ['correct', 'Speaker_full', 'Subject'], as_index=False
            ).agg(
                variability=('variability', 'mean'),
                numCorrect=('accuracy', 'sum'),
                numWord=('accuracy', 'count')
            )
            test_agg['numIncorrect'] = (test_agg['numWord'] - test_agg['numCorrect']).clip(lower=0)
            test_agg = test_agg.dropna(subset=['variability'])
            
            # Scale test using TRAINING statistics
            test_agg['variability_scaled'] = (test_agg['variability'] - train_mean) / (2 * train_sd)
            
            test_agg.rename(columns={
                'correct': 'Keyword',
                'Speaker_full': 'TestTalker',
                'Subject': 'SubjectID'
            }, inplace=True)
            
            ro.globalenv['r_test'] = pandas2ri.py2rpy(test_agg)
            ro.r("""
                r_test$Keyword   <- factor(r_test$Keyword)
                r_test$TestTalker <- factor(r_test$TestTalker)
                r_test$SubjectID <- factor(r_test$SubjectID)
                model_test <- glmer(
                    cbind(numCorrect, numIncorrect) ~ variability_scaled + (1 + variability_scaled | SubjectID),
                    data=r_test, family=binomial(link="logit"),
                    control=glmerControl(optimizer="bobyqa", optCtrl=list(maxfun=1e5)))
                z_test  <- summary(model_test)$coefficients[2, 3]
                ll_test <- as.numeric(logLik(model_test))
            """)
            
            return {
                'z_train': z_train,
                'z_test': ro.globalenv['z_test'][0],
                'poll_train': ro.globalenv['ll_train'][0] / (train_agg['numCorrect'].sum() + train_agg['numIncorrect'].sum()),
                'poll_test':  ro.globalenv['ll_test'][0]  / (test_agg['numCorrect'].sum()  + test_agg['numIncorrect'].sum()),
            }
    except Exception as e:
        print(f"  GLMM Exception: {e}")
        return 999.0 if purpose == 'optimize' else None
    return None


def process_single_layer_variability_nygaard(layer_key, df_test, h5_path, method, subject_map, subject_words_map):
    """
    Process a single layer: optimize tau, evaluate per fold.
    """
    try:
        # Load and standardize
        layer_data, speakers = load_nygaard_layer(h5_path, layer_key)
        std_data = standardize_nygaard_layer(layer_data)
        
        folds = sorted(df_test['fold'].unique())
        diagnostic_results, best_taus = [], []
        
        # Phase 1: Optimize tau per fold
        for f in folds:
            train_idx = df_test['fold'] != f
            
            def objective(tau_val):
                train_df = df_test[train_idx].copy()
                df_var = precompute_layer_variability_nygaard(
                    train_df, std_data, tau_val, method, subject_map, subject_words_map
                )
                return run_glmm_variability_logic_nygaard(tau_val, df_var, None, 'optimize')
            
            res = minimize_scalar(objective, bounds=(0.5, 4.0), method='bounded')
            best_tau = res.x
            best_taus.append(best_tau)
            
            # Evaluate with best tau
            train_df = df_test[train_idx].copy()
            test_df  = df_test[~train_idx].copy()
            train_var = precompute_layer_variability_nygaard(
                train_df, std_data, best_tau, method, subject_map, subject_words_map
            )
            test_var = precompute_layer_variability_nygaard(
                test_df, std_data, best_tau, method, subject_map, subject_words_map
            )
            
            metrics = run_glmm_variability_logic_nygaard(best_tau, train_var, test_var, 'evaluate')
            if metrics:
                diagnostic_results.append({
                    'layer': layer_key, 'fold': f, 'type': 'diagnostic', 'tau': best_tau,
                    'z_train': metrics['z_train'], 'z_test': metrics['z_test'],
                    'poll_train': metrics['poll_train'], 'poll_test': metrics['poll_test'],
                    'optimism': (metrics['poll_train'] - metrics['poll_test']) / abs(metrics['poll_train']),
                })
        
        # Phase 2: Corrected evaluation with mean tau
        mean_tau = np.mean(best_taus) if best_taus else 2.0
        corrected_results = []
        for f in folds:
            train_df = df_test[df_test['fold'] != f].copy()
            test_df  = df_test[df_test['fold'] == f].copy()
            train_var = precompute_layer_variability_nygaard(
                train_df, std_data, mean_tau, method, subject_map, subject_words_map
            )
            test_var = precompute_layer_variability_nygaard(
                test_df, std_data, mean_tau, method, subject_map, subject_words_map
            )
            
            metrics = run_glmm_variability_logic_nygaard(mean_tau, train_var, test_var, 'evaluate')
            if metrics:
                corrected_results.append({
                    'layer': layer_key, 'fold': f, 'type': 'corrected', 'tau': mean_tau,
                    'z_train': metrics['z_train'], 'z_test': metrics['z_test'],
                    'poll_train': metrics['poll_train'], 'poll_test': metrics['poll_test'],
                    'optimism': (metrics['poll_train'] - metrics['poll_test']) / abs(metrics['poll_train']),
                })
        
        results_df = pd.DataFrame(diagnostic_results + corrected_results)
        return results_df, layer_key
        
    except Exception as e:
        return f"EXCEPTION in {layer_key}: {e}\n{traceback.format_exc()}"


def run_variability_pipeline_nygaard(layers_list, df_test, h5_path, method, subject_map, subject_words_map, n_jobs=-1):
    """Orchestrate parallel execution across layers."""
    import time
    print(f"Starting {method} analysis for {len(layers_list)} layers...")
    start = time.time()
    
    tasks = [
        delayed(process_single_layer_variability_nygaard)(
            k, df_test, h5_path, method, subject_map, subject_words_map
        )
        for k in layers_list
    ]
    raw_results = Parallel(n_jobs=n_jobs, verbose=5)(tasks)
    
    valid_results = []
    for item in raw_results:
        if isinstance(item, str):
            print(item)
            continue
        if item is None:
            continue
        res_df, layer_key = item
        if res_df is not None and not res_df.empty:
            valid_results.append(res_df)
    
    all_layers_results = pd.concat(valid_results, ignore_index=True) if valid_results else pd.DataFrame()
    print(f"Done in {(time.time() - start):.2f}s")
    return all_layers_results


## 5. Run Variability Pipeline

Execute the variability analysis for all three methods across all HuBERT layers.


In [ ]:
METHODS = ['BetweenWord', 'WithinWord', 'Word_Order']

all_method_results = {}

for method in METHODS:
    print(f"\n{'='*60}")
    print(f"Running GLMM Analysis for method: {method}")
    print(f"{'='*60}")
    
    glmm_res = run_variability_pipeline_nygaard(
        layers_list, df_test, H5_PATH, method, subject_map, subject_words_map, n_jobs=-1
    )
    
    if not glmm_res.empty:
        out_path = f"nygaard19_tsne_ft_variability_glmm_{method}.csv"
        glmm_res.to_csv(out_path, index=False)
        all_method_results[method] = glmm_res
        print(f"Results saved to {out_path}")
    else:
        print(f"WARNING: No results for {method}")

print(f"\n{'='*60}")
print("All methods complete!")
print(f"{'='*60}")


## 6. Visualization

Plot the predictive power (z-test) of each variability measure across HuBERT layers.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def extract_layer_num(layer_name):
    """Sort key: CNN layers first (by number), then Transformer layers."""
    if 'cnn' in layer_name:
        return int(layer_name.split('_')[1])
    elif 'tr' in layer_name:
        return 100 + int(layer_name.split('_')[1])
    return 0

def format_label(l):
    if l.startswith('cnn_'):
        return f"CNN-{l.split('_')[1]}"
    elif l.startswith('tr_'):
        return f"TR-{l.split('_')[1]}"
    return l

# ── Load results (from CSV if not in memory) ──
for method in METHODS:
    csv_path = f"nygaard19_tsne_ft_variability_glmm_{method}.csv"
    if method not in all_method_results and os.path.exists(csv_path):
        all_method_results[method] = pd.read_csv(csv_path)

# ── Plot each method ──
fig, axes = plt.subplots(1, len(METHODS), figsize=(7 * len(METHODS), 6), sharey=True)
if len(METHODS) == 1:
    axes = [axes]

for ax, method in zip(axes, METHODS):
    if method not in all_method_results:
        ax.set_title(f"{method} (no data)")
        continue
    
    df_plot = all_method_results[method]
    df_corr = df_plot[df_plot['type'] == 'corrected'].copy()
    
    if df_corr.empty:
        ax.set_title(f"{method} (no corrected results)")
        continue
    
    df_corr['layer_order'] = df_corr['layer'].apply(extract_layer_num)
    df_corr = df_corr.sort_values('layer_order')
    df_corr['Feature space'] = df_corr['layer'].apply(format_label)
    
    order = df_corr.sort_values('layer_order')['Feature space'].unique()
    
    sns.pointplot(
        data=df_corr, x='Feature space', y='z_test', order=order,
        color='black', markers='o', scale=1.0,
        errorbar=('ci', 95), capsize=0.1, linestyle='-',
        err_kws={'linewidth': 1.5}, ax=ax
    )
    
    ax.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.6)
    ax.axhline(1.96, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='p=0.05')
    ax.set_title(f"Nygaard (AN19) - {method}", fontsize=13, fontweight='bold')
    ax.set_xlabel("Representation Layer", fontsize=11)
    ax.set_ylabel("Z-Value (Test)" if ax == axes[0] else "", fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.4)

plt.suptitle("Variability Predictive Power Across HuBERT Layers (Nygaard AN19)",
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/nygaard19_variability_across_layers.png', dpi=200, bbox_inches='tight')
plt.show()


## 7. Ceiling-Normalized Visualization

Normalize variability z-test scores relative to the behavioral noise ceiling (Jaeger Self-Predictability Method).


In [ ]:
# ── Compute noise ceiling ──
ceiling_vals = compute_jaeger_ceiling_nygaard(df_test)
ceil_mean = np.mean(ceiling_vals)
ceil_sem  = np.std(ceiling_vals, ddof=1) / np.sqrt(len(ceiling_vals))

print(f"Ceiling: z = {ceil_mean:.4f} (SEM: {ceil_sem:.4f})")
print(f"Per-fold z: {ceiling_vals}")

# ── Plot with ceiling normalization ──
fig, axes = plt.subplots(1, len(METHODS), figsize=(7 * len(METHODS), 6), sharey=True)
if len(METHODS) == 1:
    axes = [axes]

for ax, method in zip(axes, METHODS):
    if method not in all_method_results:
        continue
    
    df_plot = all_method_results[method]
    df_corr = df_plot[df_plot['type'] == 'corrected'].copy()
    
    if df_corr.empty:
        continue
    
    df_corr['layer_order'] = df_corr['layer'].apply(extract_layer_num)
    df_corr = df_corr.sort_values('layer_order')
    df_corr['Feature space'] = df_corr['layer'].apply(format_label)
    df_corr['percent_ceiling'] = (df_corr['z_test'].abs() / ceil_mean) * 100
    
    order = df_corr.sort_values('layer_order')['Feature space'].unique()
    
    sns.pointplot(
        data=df_corr, x='Feature space', y='percent_ceiling', order=order,
        color='black', markers='o', scale=1.0,
        errorbar=('ci', 95), capsize=0.1, linestyle='-',
        err_kws={'linewidth': 1.5}, ax=ax
    )
    
    ceil_sem_pc = (ceil_sem / ceil_mean) * 100
    ax.axhspan(100.0 - ceil_sem_pc, 100.0 + ceil_sem_pc, color='#e0e0e0', alpha=0.8, zorder=0)
    ax.axhline(100, color='black', linestyle='--', linewidth=1.5, zorder=1)
    ax.axhline((1.96 / ceil_mean) * 100, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='p=0.05')
    
    ax.set_title(f"Nygaard (AN19) - {method}", fontsize=13, fontweight='bold')
    ax.set_xlabel("Representation Layer", fontsize=11)
    ax.set_ylabel("% Behavioral Ceiling" if ax == axes[0] else "", fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylim(0, 120)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.4)

plt.suptitle("Variability (% Ceiling) Across HuBERT Layers (Nygaard AN19)",
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/nygaard19_variability_percent_ceiling.png', dpi=200, bbox_inches='tight')
plt.show()
